# 2D Finite Element Simulation for Generalized Poisson Equation
This Jupyter notebook implements from scratch a 2D finite element code for the generalized Poisson equation. 

In [7]:
"""
    generate_mesh(filename, h; verbosity=0)

    Generates a mesh with mesh size `h` using GMSH with a `.geo` input file `filename`.
    The mesh is saved in msh2 format as `filename.msh`.
    Note that this function requires gmsh to be callable via the command line!
"""
function generate_mesh(filename, h; verbosity=0)
    # -2: 2d mesh generation
    # -clscale sets the mesh size factor, that is, the average edge length h
    # -format: we are here using the slightly older MSH2 format
    # -v: level of verbosity
    # run gmsh -help in the command line to learn about the details 
    # or consult the online help
    run(`gmsh $filename -2 -clscale $h -format msh2 -v $verbosity`)
end

generate_mesh

In [8]:
using StaticArrays
using DelimitedFiles

"""
    PhysicalTag{I <: Integer, S <: String}

Type to store a physical tag of a GMSH mesh.

# Fields
- `dimension::I`: stores the dimensionality of entities (1D, 2D, 3D etc.)
- `tagname::S`: a string with the name of tag
"""
struct PhysicalTag{I <: Integer, S <: String}
    dimension::I
    tagname::S
end

function tagname(physicaltag::PhysicalTag)
   return physicaltag.tagname
end

"""
    Mesh{I,S,F}

Type to store GMSH mesh.

# Fields
- `physicaltags::Vector{PhysicalTag{I,S}}`: stores the physical tags
- `vertices::Vector{SVector{3, F}}`: stores the vertices
- `lines::Vector{SVector{2, I}}`: stores the lines
- `lines_physicaltags::Vector{SVector{2, I}}`: stores the physical tags for each line segment
- `triangles::Vector{SVector{3, I}}`: stores the triangles of the mesh
"""
mutable struct Mesh{I <: Integer, S <: String, F <: Real}
    physicaltags::Vector{PhysicalTag{I,S}}
    vertices::Vector{SVector{3, F}}
    lines::Vector{SVector{2, I}}
    lines_physicaltags::Vector{I}
    triangles::Vector{SVector{3, I}}
end

function vertices(Γ::Mesh)
    return Γ.vertices
end

function numvertices(Γ::Mesh)
    return length(Γ.vertices)
end

function lines(Γ::Mesh)
    return Γ.lines
end

function triangles(Γ::Mesh)
    return Γ.triangles
end

function numtriangles(Γ::Mesh)
    return length(Γ.triangles)
end

function physicaltags(Γ::Mesh)
   return  Γ.physicaltags
end

function read_gmsh(fn::AbstractString)
    open(fn, "r") do io  
        return read_gmsh(io)
    end
end
    
function read_gmsh(io)
    # The '|>' operator is documented in the 
    # 'Essentials' section of the Julia manual
    thisLine = io |> readline |> strip 
    
    while thisLine != "\$PhysicalNames" # We need to escape the '$'
        thisLine = io |> readline |> strip        end

    thisLine = io |> readline |> strip

    s = split(thisLine)
    NP = parse(Int, s[1])
    P = PhysicalTag{Int, String}
    p = Vector{P}(undef, NP)

    # Read in physical tags
    for i in 1:NP
        thisLine = io |> readline |>  strip
        d = readdlm(IOBuffer(thisLine), Any)
        p[i] = P(d[1], d[3])
    end
    
    while thisLine != "\$Nodes" # We need to escape the '$'
        thisLine = io |> readline |> strip        end

    thisLine = io |> readline |> strip
    s = split(thisLine)
    NV = parse(Int, s[1])
    
    P = SVector{3,Float64}
    v = Vector{P}(undef,NV)
    # Read in vertices
    for i in 1:NV
        thisLine = io |> readline |>  strip
        d = readdlm(IOBuffer(thisLine), Float64)
        # Format of each line: node-number x-coord y-coord z-coord
        v[i] = P(d[2], d[3], d[4])
    end

    while thisLine != "\$Elements"
        thisLine = io |> readline |> strip
    end

    thisLine = io |> readline |> strip

    s = split(thisLine)
    NE = parse(Int, s[1])

    thisLine = io |> readline |> strip
    I2 = SVector{2,Int}
    I3 = SVector{3,Int}
    l = Vector{I2}(undef,NE)
    lp = Vector{Int}(undef,NE)
    f = Vector{I3}(undef,NE)
    lc = 0 # Lines' counter
    fc = 0 # triangles' counter
    # Read in elements
    for i=1:NE
        # Format of each line: 
        # elm-number elm-type number-of-tags < tag > … node-number-list
        d = readdlm(IOBuffer(thisLine), Int)
        thisLine = io |> readline |> strip
        
        if d[2] == 1 # elm-type must be 3-node triangle
            l[lc+=1] = d[end-1:end]
            
            if d[3] == 2 # Ensure that there are two tags, the first being the physical tag
                lp[lc] = d[4] # Physical tag is always the first tag
            else
                error("We require each part of the boundary to be associated with a physical group")
            end
        end
        
        if d[2] == 2
            f[fc+=1] = d[end-2:end] # Last three entries per line are the nodes of the triangle
        end
    end
    resize!(l,lc)
    resize!(lp,lc)
    resize!(f,fc)

    return Mesh(p, v, l, lp, f)
end

function integertype(Γ)
    return typeof(Γ).parameters[1]
end

integertype (generic function with 1 method)

In [9]:
using SparseArrays
using IterativeSolvers

"""
    computecapacitance(h; usesparsematrices=false)

Computes the capacitance of a structure meshed with mesh size `h`.
The structure is assumed to have an inner and an outer boundary,
where the outer boundary will be grounded.
"""
function computecapacitance(h; usesparsematrices=false)
    # Generate a mesh using GMSH and
    # a geometry input file called capacitors_rectangular.geo
    generate_mesh("capacitors_rectangular.geo", h)
    
    # Read the mesh
    Ω = read_gmsh("capacitors_rectangular.msh")

    # Generate arrays similar to the ones in the textbook
    # Similar: because we use nested vectors
    # (here: Vector of SVectors)
    # while the textbook uses matrices (which is more old school)
    no2xy = [SVector{2}(elem[1], elem[2]) for (idx, elem) in enumerate(vertices(Ω))]
    noInt = integertype(Ω)[]
    noExt = integertype(Ω)[]

    for (i, elem) in enumerate(Ω.lines)
        if tagname(Ω.physicaltags[Ω.lines_physicaltags[i]]) ==  "InnerConductor"
            push!(noInt, elem[1], elem[2])
        elseif tagname(Ω.physicaltags[Ω.lines_physicaltags[i]]) ==  "OuterConductor"
            push!(noExt, elem[1], elem[2])
        else
            error("Encountered unexpected boundary: ", Ω.physicaltags[Ω.lines_physicaltags[i]])
        end    
    end
    unique!(noInt) # Remove the duplicate entries
    unique!(noExt)
    
    # Assemble the matrix A
    if usesparsematrices
        A = assemblesparse(Ω)
    else
        A = assembledense(Ω)
    end
    
    # Get the indices of the boundary and non-boundary nodes
    no_ess = vcat(noInt, noExt) # Boundary nodes for Dirichlet BC
    no_all = 1:numvertices(Ω)
    no_nat = setdiff(no_all, no_ess) # Interior nodes


    # Pick out the parts of the matrix and the vectors
    # needed to solve the problem.
    A_ess    = A[no_nat, no_ess]
    A_nat    = A[no_nat, no_nat]

    # Voltage between inner and outer conductor.
    U = 1

    z        = zeros(length(no_all))
    z[noInt] = U*ones(length(noInt)) 
    z_ess    = z[no_ess]

    # Solve the system of linear equations
    println("Solve system with number of unknowns: ", numvertices(Ω))
    z_nat = solve(A_nat, - A_ess*z_ess)  

    # Build up the total solution.
    z = zeros(numvertices(Ω))
    z[no_ess] = z_ess
    z[no_nat] = z_nat

    # Compute the capacitance.
    ε0 = 8.8541878128e-12 # Permittivity in vacuum
    W = 0.5*ε0*(z'*A*z) # Recall the homework assignment H 2.4
    C = 2*W/U^2

    println("C per unit length [pF/m] = " * string(C/1e-12))
    return C, Ω, z
end

"""
    cmpelmtx(xy)

This function computes the element matrix for one triangle, where `xy` is
a vector containing the coordinates of the nodes of the triangle.
The function follows the logic of the function CmpElMtx in the textbook,
but is adapted to the use of nested vectors instead of a matrix.

# Arguments
- `xy`: the coordinates of the nodes of the triangle

# Returns
- `Matrix`: element matrix corresponding to the laplace-operator
"""
function cmpelmtx(xy)
    # Edges
    s1 = xy[3]-xy[2]
    s2 = xy[1]-xy[3]
    s3 = xy[2]-xy[1]

    # Area of the triangle 
    Atot = 0.5*(s2[1]*s3[2]-s2[2]*s3[1])

    # Check if area is negative (nodes given counterclockwise)

    if Atot < 0
        error("The nodes of the element given in wrong order")
    end  

    # Compute the gradient of the vectors.
    grad_phi1e = [-s1[2]; s1[1]]/(2*Atot)
    grad_phi2e = [-s2[2]; s2[1]]/(2*Atot)
    grad_phi3e = [-s3[2]; s3[1]]/(2*Atot)

    grad_phi = [grad_phi1e grad_phi2e grad_phi3e]

    Ae = zeros(3, 3)
    
    # Compute all the integrals for this particular element.
    for iIdx = 1:3
        for jIdx = 1:3
            Ae[iIdx, jIdx] = grad_phi[:, iIdx]' * grad_phi[:, jIdx] * Atot
        end
    end
    return Ae
end

"""
    assembledense(Ω::Mesh{I,S,F}) where {I,S,F}

Returns the asssembled system matrix as a dense matrix.
"""
function assembledense(Ω::Mesh{I,S,F}) where {I,S,F}
    A = zeros(F, numvertices(Ω), numvertices(Ω))

    for no in triangles(Ω)
        xy = [vertices(Ω)[elem] for elem in no] # no2xy[:, no] 

        # Compute the element matrix and add
        # the contribution to the global matrix.
        A_el = cmpelmtx(xy)
        A[no,no] += A_el
    end
    
    return A
end

"""
    assemblesparse(Ω::Mesh{I,S,F}) where {I,S,F}

Returns the asssembled system matrix as a sparse matrix.
"""
function assemblesparse(Ω::Mesh{I,S,F}) where {I,S,F}
    # Use sparse matrices:
    # Store coordinates and matrix element entry
    # in three vectors
    rows = I[]
    columns = I[]
    elems = F[]

    for no in triangles(Ω)
        xy = [vertices(Ω)[elem] for elem in no] # no2xy[:, no] 

        # Compute the element matrix and add
        # the contribution to the global matrix.
        A_el = cmpelmtx(xy)
        for (i, row) in enumerate(no)
            for (j, col) in enumerate(no)
                push!(rows, row)
                push!(columns, col)
                push!(elems, A_el[i,j])
            end
        end
    end
    
    return sparse(rows, columns, elems)
end

function solve(A, b)
    return A\b
end

function solve(A::SparseMatrixCSC, b)
    z_nat, history = cg(A, b, reltol=1e-4, verbose=true, log=true) # Note that b = 0
    return z_nat
end

solve (generic function with 2 methods)

In [10]:
@time computecapacitance(0.1)

/usr/bin/env: ‘python’: No such file or directory


LoadError: failed process: Process(`[4mgmsh[24m [4mcapacitors_rectangular.geo[24m [4m-2[24m [4m-clscale[24m [4m0.1[24m [4m-format[24m [4mmsh2[24m [4m-v[24m [4m0[24m`, ProcessExited(127)) [127]


In [6]:
@time computecapacitance(0.1, usesparsematrices=true)

Solve system with number of unknowns: 4763
  1	7.37e+00
  2	5.22e+00
  3	4.01e+00
  4	3.20e+00
  5	2.73e+00
  6	2.47e+00
  7	2.09e+00
  8	1.83e+00
  9	1.66e+00
 10	1.51e+00
 11	1.37e+00
 12	1.27e+00
 13	1.18e+00
 14	1.10e+00
 15	1.04e+00
 16	9.74e-01
 17	9.08e-01
 18	8.51e-01
 19	8.10e-01
 20	7.77e-01
 21	7.32e-01
 22	5.74e-01
 23	2.53e-01
 24	1.51e-01
 25	2.03e-01
 26	1.62e-01
 27	1.70e-01
 28	1.45e-01
 29	1.34e-01
 30	1.21e-01
 31	1.02e-01
 32	8.98e-02
 33	7.62e-02
 34	6.68e-02
 35	6.18e-02
 36	5.41e-02
 37	4.74e-02
 38	4.35e-02
 39	3.78e-02
 40	3.23e-02
 41	2.78e-02
 42	2.35e-02
 43	1.97e-02
 44	1.70e-02
 45	1.36e-02
 46	1.13e-02
 47	9.75e-03
 48	7.99e-03
 49	6.08e-03
 50	4.86e-03
 51	4.17e-03
 52	3.41e-03
 53	2.90e-03
 54	2.45e-03
 55	2.09e-03
 56	1.76e-03
 57	1.45e-03
 58	1.21e-03

C per unit length [pF/m] = 90.80527637112019
  1.006121 seconds (862.45 k allocations: 608.662 MiB, 17.28% gc time, 47.65% compilation time)


(9.080527637112019e-11, Mesh{Int64, String, Float64}(PhysicalTag{Int64, String}[PhysicalTag{Int64, String}(1, "InnerConductor"), PhysicalTag{Int64, String}(1, "OuterConductor"), PhysicalTag{Int64, String}(2, "Interior")], SVector{3, Float64}[[0.5, 0.5, 0.0], [-0.5, 0.5, 0.0], [-0.5, -0.5, 0.0], [0.5, -0.5, 0.0], [1.0, 1.0, 0.0], [-1.0, 1.0, 0.0], [-1.0, -1.0, 0.0], [1.0, -1.0, 0.0], [0.4722222222222993, 0.5, 0.0], [0.4444444444445986, 0.5, 0.0]  …  [-0.9124293131527106, 0.4645739227121639, 0.0], [-0.9121695401985424, 0.5207726759087186, 0.0], [-0.3539601033162041, 0.912106675216429, 0.0], [0.9154929029275908, -0.9613178917798607, 0.0], [0.3022060638015, 0.6745790639704952, 0.0], [-0.2001041967864747, 0.6623240160221716, 0.0], [-0.2123334447265662, 0.9119163431087431, 0.0], [0.3744846177204625, 0.6899966816522365, 0.0], [0.4792892734127074, -0.9321505412653301, 0.0], [0.5268001085846706, -0.9321751595324693, 0.0]], SVector{2, Int64}[[1, 9], [9, 10], [10, 11], [11, 12], [12, 13], [13, 14

In [5]:
using Plots

C, Ω, z = computecapacitance(0.5)

x = [vertex[1] for vertex in vertices(Ω)]
y = [vertex[2] for vertex in vertices(Ω)]

plot(
    x,
    y,
    z,
    st=:surface,
    xlabel="x",
    ylabel="y",
    zlabel="Φ",
    title="Plot of the solution of ΔΦ=0 with Dirichlet BCs"
)

LoadError: IOError: could not spawn `gmsh capacitors_rectangular.geo -2 -clscale 0.5 -format msh2 -v 0`: no such file or directory (ENOENT)

In [9]:
# And here a bit of convergence analysis: first let's use dense matrices

function estimate_p(h, I)
    return log((I[end-2] - I[end-1])/(I[end-1] - I[end]))/log(h[end-2]/h[end-1])
end

h_vals = [0.4; 0.2; 0.1]
I = [computecapacitance(h)[1] for h in h_vals] # First return argument is the capacitance

p = estimate_p(h_vals, I)

Solve system with number of unknowns: 335
C per unit length [pF/m] = 91.8620400301389
Solve system with number of unknowns: 1217
C per unit length [pF/m] = 91.1108023520951
Solve system with number of unknowns: 4763
C per unit length [pF/m] = 90.80523676214204


1.2977873334466647

In [10]:
# And now with sparse matrices

h_vals = [0.1; 0.05; 0.025]
I = [computecapacitance(h, usesparsematrices=true)[1] for h in h_vals]

p = estimate_p(h_vals, I)

Solve system with number of unknowns: 4763
  1	7.37e+00
  2	5.22e+00
  3	4.01e+00
  4	3.20e+00
  5	2.73e+00
  6	2.47e+00
  7	2.09e+00
  8	1.83e+00
  9	1.66e+00
 10	1.51e+00
 11	1.37e+00
 12	1.27e+00
 13	1.18e+00
 14	1.10e+00
 15	1.04e+00
 16	9.74e-01
 17	9.08e-01
 18	8.51e-01
 19	8.10e-01
 20	7.77e-01
 21	7.32e-01
 22	5.74e-01
 23	2.53e-01
 24	1.51e-01
 25	2.03e-01
 26	1.62e-01
 27	1.70e-01
 28	1.45e-01
 29	1.34e-01
 30	1.21e-01
 31	1.02e-01
 32	8.98e-02
 33	7.62e-02
 34	6.68e-02
 35	6.18e-02
 36	5.41e-02
 37	4.74e-02
 38	4.35e-02
 39	3.78e-02
 40	3.23e-02
 41	2.78e-02
 42	2.35e-02
 43	1.97e-02
 44	1.70e-02
 45	1.36e-02
 46	1.13e-02
 47	9.75e-03
 48	7.99e-03
 49	6.08e-03
 50	4.86e-03
 51	4.17e-03
 52	3.41e-03
 53	2.90e-03
 54	2.45e-03
 55	2.09e-03
 56	1.76e-03
 57	1.45e-03
 58	1.21e-03

C per unit length [pF/m] = 90.80527637112019
Solve system with number of unknowns: 18010
  1	1.00e+01
  2	7.01e+00
  3	5.47e+00
  4	4.36e+00
  5	3.57e+00
  6	3.08e+00
  7	2.74e+00
  8	2.45e+00
  9	2.19e

1.2739613458591874